# Day 12 — Python Modules, Type Hints & pytest

Modules: `reader.py`, `validator.py`, `transformer.py`  
Dataset: Olist Brazilian E-Commerce (order_items.csv)

In [ ]:
# Parameters
INPUT_FOLDER = "../data"
ORDERS_FILE  = f"{INPUT_FOLDER}/orders.csv"
ITEMS_FILE   = f"{INPUT_FOLDER}/order_items.csv"
MAPPING_FILE = "../config/column_mapping.json"

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../src").resolve()))

%load_ext autoreload
%autoreload 2

## Task 3 — read_csv(path) → list of dicts

In [ ]:
from reader import read_csv

orders = read_csv(ORDERS_FILE)
print(f"orders.csv rows: {len(orders)}")
print("First row:", orders[0])

## Task 4 — Convert column names to snake_case

In [ ]:
from validator import to_snake_case

sample_names = ["OrderID", "freight value", "order-status", "customerName"]
for name in sample_names:
    print(f"{name!r:30s} -> {to_snake_case(name)!r}")

## Task 5 — Column mapping from JSON config

In [ ]:
from validator import load_column_mapping, apply_column_mapping

mapping = load_column_mapping(MAPPING_FILE)
print("Column mapping loaded:")
for orig, standard in mapping.items():
    print(f"  {orig:45s} -> {standard}")

items = read_csv(ITEMS_FILE)
items_mapped = apply_column_mapping(items, mapping)
print("\nMapped columns:", list(items_mapped[0].keys()))

## Task 6 — Convert price, freight to float; timestamps to datetime

In [ ]:
from transformer import convert_types

FLOAT_COLS    = ["price", "freight_value"]
DATETIME_COLS = ["shipping_limit_at"]

items_typed = convert_types(items_mapped, FLOAT_COLS, DATETIME_COLS)

row = items_typed[0]
print(f"price type       : {type(row['price']).__name__}  value={row['price']}")
print(f"freight type     : {type(row['freight_value']).__name__}  value={row['freight_value']}")
print(f"shipping_limit_at: {type(row['shipping_limit_at']).__name__}  value={row['shipping_limit_at']}")

## Task 7 — Order totals per order_id

In [ ]:
from transformer import calculate_order_totals

totals = calculate_order_totals(items_typed)
print(f"Total unique orders : {len(totals)}")
top5 = sorted(totals.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nTop 5 orders by spend:")
for oid, total in top5:
    print(f"  {oid}  R$ {total:.2f}")

## Task 8 — Run pytest from the notebook

In [ ]:
!python -m pytest ../tests/test_day12.py -v